# Generate report for tax purposes

The template for celiac tax credit is usually stuctured like this: 
item type, quantity, non-gf average cost, average gf-cost, average incremental cost, total claim

Mapping: 
- item type: reference item name
- quantity: count expenses group by reference item
- non-gf average cost: reference item price
- average gf cost: ... see below
- average incremental cost: difference of non-gf and gf
- total: count * difference

Note on average gf cost: In my case I don't count instances of expenses (e.g. 1x hot dog buns) but I adjust it to quantity/weight (e.g. 4 hot dog buns). To compare apples to apples, I need the expense amount adjusted to the weight/count of the reference item. That means that for each expense I need to calculate the price the item would be if it was the weight of the reference item but with the price per weight of the expense using the weight of product associated with the expense. 

In [138]:
%pip install -q \
    psycopg2-binary \
    pandas


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [139]:
import psycopg2
import pandas as pd

In [140]:
# Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="receipts_app",
    user="postgres",
    password="postgres",
    options="-c search_path=receipts_app"
)

# Pre-flight

In [141]:
cur = conn.cursor()


## Check for missing price proofs

In [142]:
with open('../nifi_product_price_proof_request.sql', 'r') as f:
    query = f.read()
cur.execute(query)

# Fetch results
results = cur.fetchall()

print(f"Found {len(results)} reference items needing price proofs:")
for row in results:
    print(f"Reference Item ID: {row[0]}")

Found 0 reference items needing price proofs:


# Report

## General clean-up notes

- I noticed a few $0 expenses, manually investigated and fixed
- Picking up mistakes like a $999 item (decimal point not dected in OCR)
- Conversion on gum was incorrect (fixed it by setting it the same in the product)

## Check for missing conversions

In [143]:
# with open('../report.sql', 'r') as f:
#     query = f.read()
# cur.execute(query)

# # Fetch results
# results = cur.fetchall()

# df = pd.DataFrame(results, columns=[
#     'expense_id',
#     'receipt_id',
#     'price_each',
#     'quantity',
#     'total_price',
#     'expense_date',
#     'product_id',
#     'product_name',
#     'product_weight',
#     'product_unit_of_measure',
#     'reference_item_id',
#     'reference_item_name',
#     'price_proof_id',
#     'price_proof_name',
#     'proof_price',
#     'proof_quantity',
#     'proof_unit_of_measure',
#     'price_proof_date',
#     'equivanlent_base_price',
#     'product_cost_difference',
#     'gf_total'
# ])
# df

In [144]:
# pd.set_option('display.max_columns', 100)
# df

In [145]:
# df[df['reference_item_id'] == 4]
# df[df['reference_item_id'] == 4]['quantity'].sum()

## CRA-style report

In [146]:
with open('../report.sql', 'r') as f:
    query = f.read()
cur.execute(query)

# Fetch results
results = cur.fetchall()

df = pd.DataFrame(results, columns=[
    'reference_item_id', 
	'reference_item_per_id',
	'quantity',
	'non_gf_average_cost',
	'gf_average_cost',
	'non_gf_average_cost',
    'avg_incr_cost',
    'total_cost'
])

# df = df.drop(columns=['reference_item_id'])
cost_columns = list(set(col for col in df.columns if col.endswith('_cost')))
for col in cost_columns:
    df[col] = df[col].astype(float).map('${:,.2f}'.format)

df

,reference_item_id,reference_item_per_id,quantity,non_gf_average_cost,gf_average_cost,non_gf_average_cost,avg_incr_cost,total_cost
0,4,Candy,22.0,$1.66,$5.44,$1.66,$3.78,$83.10
1,5,Cereal,21.0,$5.52,$10.50,$5.52,$4.98,$104.49
2,6,Chia seeds,3.0,$4.49,$9.66,$4.49,$5.17,$15.50
3,7,Chips,18.0,$1.98,$6.04,$1.98,$4.06,$73.09
4,8,Chocolate,11.0,$1.10,$5.29,$1.10,$4.19,$46.12
5,9,Chocolate chips,6.0,$2.86,$7.50,$2.86,$4.64,$27.86
6,10,Cookies,5.0,$0.99,$6.09,$0.99,$5.10,$25.50
7,12,Crackers,11.0,$1.00,$5.94,$1.00,$4.93,$54.27
8,13,Flour,35.0,$1.03,$7.76,$1.03,$6.73,$235.47
9,14,Granola,20.0,$2.44,$6.90,$2.44,$4.46,$89.16


In [147]:
total_cost_sum = df['total_cost'].replace({'\$': '', ',': ''}, regex=True).astype(float).sum()
print(f'Total of total_cost column: ${total_cost_sum:,.2f}')


Total of total_cost column: $3,028.68


<>:1: SyntaxWarning: invalid escape sequence '\$'
<>:1: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_32238/1169038756.py:1: SyntaxWarning: invalid escape sequence '\$'
  total_cost_sum = df['total_cost'].replace({'\$': '', ',': ''}, regex=True).astype(float).sum()


## LLM verify

In [148]:
prompt = f"""You are an experienced tax auditor specializing in expense deductions, particularly for celiac disease tax credits in Canada. Your task is to review the following expense report and assess each item's eligibility for the celiac disease tax credit.

For each listed item:
1. Verify if the gluten-free product and its regular counterpart are properly categorized
2. Analyze if the price differential between the gluten-free version and regular version is reasonable based on market standards
3. Flag any items where the price difference appears suspicious (either too large or too small)
4. Confirm that only the price difference between gluten-free and regular products is being claimed, not the full cost
5. Check if the items qualify as "medical food" under tax regulations (essential for basic nutrition, not just specialty or luxury items)

Please provide your analysis in a structured format:
- APPROVED: Items that appear legitimate and properly documented
- REQUIRES CLARIFICATION: Items where additional information is needed
- REJECTED: Items that clearly do not qualify for the tax credit

For any flagged items, explain specifically what concerns you have and what additional documentation would be required to approve the claim.

Here is the expense report to review:

{df[['reference_item_per_id', 'quantity', 'non_gf_average_cost', 'gf_average_cost']].to_csv(index=False)}
"""
print(prompt)




You are an experienced tax auditor specializing in expense deductions, particularly for celiac disease tax credits in Canada. Your task is to review the following expense report and assess each item's eligibility for the celiac disease tax credit.

For each listed item:
1. Verify if the gluten-free product and its regular counterpart are properly categorized
2. Analyze if the price differential between the gluten-free version and regular version is reasonable based on market standards
3. Flag any items where the price difference appears suspicious (either too large or too small)
4. Confirm that only the price difference between gluten-free and regular products is being claimed, not the full cost
5. Check if the items qualify as "medical food" under tax regulations (essential for basic nutrition, not just specialty or luxury items)

Please provide your analysis in a structured format:
- APPROVED: Items that appear legitimate and properly documented
- REQUIRES CLARIFICATION: Items where 

# Cleanup

In [149]:
# Close connection
cur.close()
conn.close()